# Task 1: Source Discovery and Data Preparation

In this task, the assigned chapter is used as the knowledge base for the Retrieval-Augmented Generation (RAG) system. According to the assignment instructions, the chapter is determined by the last digit of the student ID. Since my student ID ends with 7, the assigned chapter is Chapter 7. The PDF file for this chapter has already been downloaded and stored locally as `assets/Chapter_7.pdf`.

The main objective of this task is to prepare the chapter content so that it can later be used in both Naive RAG and Contextual Retrieval. To achieve this, the text must first be extracted from the PDF, cleaned to remove unwanted formatting noise, and then divided into manageable chunks for retrieval. In addition, at least 20 question-answer pairs must be created based strictly on the content of the assigned chapter. These QA pairs will later serve as evaluation queries for comparing the two retrieval methods.

In [1]:
# standard library imports
from pathlib import Path
import re
import json

# PDF reading library
import fitz  # PyMuPDF

# Optional display helpers
from pprint import pprint

## Defining Input and Output Paths

This section defines the location of the source PDF and the output files that will be created during preprocessing. The chapter PDF is stored in the `assets` folder, while all generated outputs are saved into an `artefacts` folder. This makes the workflow more organized and reproducible. Separate files are prepared for the raw extracted text, cleaned text, chunked text, and the final QA pairs.

In [2]:
# Define the project paths
PROJECT_ROOT = Path(".")
ASSETS_DIR = PROJECT_ROOT / "assets"
OUTPUT_DIR = PROJECT_ROOT / "artefacts"

# Create output directory if it does not exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Define the input PDF path
PDF_PATH = ASSETS_DIR / "Chapter_7.pdf"

# Define output file paths
RAW_TEXT_PATH = OUTPUT_DIR / "chapter_7_raw.txt"
CLEAN_TEXT_PATH = OUTPUT_DIR / "chapter_7_cleaned.txt"
CHUNKS_PATH = OUTPUT_DIR / "chapter_7_chunks.json"
QA_PATH = OUTPUT_DIR / "qa_pairs_chapter_7.json"

print("PDF path:", PDF_PATH)
print("Output directory:", OUTPUT_DIR)

PDF path: assets/Chapter_7.pdf
Output directory: artefacts


## Extracting Text from the PDF

The chapter content is first extracted from the PDF file so that it can be processed as plain text. This is necessary because retrieval systems operate on textual data rather than directly on PDF files. The extraction function opens the PDF using PyMuPDF and reads the text from each page sequentially. All page contents are then combined into one full document string. The raw extracted text is saved into a text file so that the original output can be inspected later if any preprocessing issues occur.

In [3]:
def extract_text_from_pdf(pdf_path: Path) -> str:
    """
    Extract text from all pages of a PDF file.

    Parameters:
        pdf_path (Path): Path to the PDF file.

    Returns:
        str: Combined text from all pages.
    """
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF file not found: {pdf_path}")

    all_pages = []

    # Open the PDF document
    with fitz.open(pdf_path) as doc:
        for page_num, page in enumerate(doc, start=1):
            page_text = page.get_text("text")
            all_pages.append(page_text)

    return "\n".join(all_pages)


# Extract raw text from the chapter PDF
raw_text = extract_text_from_pdf(PDF_PATH)

# Save raw text for inspection and reproducibility
RAW_TEXT_PATH.write_text(raw_text, encoding="utf-8")

print("Raw text length:", len(raw_text))
print("First 1000 characters of raw text:\n")
print(raw_text[:1000])

Raw text length: 91498
First 1000 characters of raw text:

Speech and Language Processing.
Daniel Jurafsky & James H. Martin.
Copyright © 2026.
All
rights reserved.
Draft of January 6, 2026.
CHAPTER
7
Large Language Models
“How much do we know at any time? Much more, or so I believe, than we
know we know.”
Agatha Christie, The Moving Finger
The literature of the fantastic abounds in inanimate objects magically endowed with
the gift of speech. From Ovid’s statue of Pygmalion to Mary Shelley’s story about
Frankenstein, we continually reinvent stories about
creating something and then having a chat with it.
Legend has it that after ﬁnishing his sculpture Moses,
Michelangelo thought it so lifelike that he tapped it
on the knee and commanded it to speak. Perhaps
this shouldn’t be surprising. Language is the mark
of humanity and sentience. conversation is the most
fundamental arena of language, the ﬁrst kind of lan-
guage we learn as children, and the kind we engage in
constantly, whether we

## Cleaning the Extracted Text

PDF-extracted text often contains noise such as inconsistent line breaks, unnecessary spaces, repeated blank lines, or isolated page numbers. These issues can negatively affect chunking and later retrieval quality. Therefore, the extracted text is cleaned before being used as the knowledge base. In this notebook, the cleaning process normalizes line breaks, removes lines that contain only page numbers, reduces repeated spaces, and compresses excessive blank lines. The goal is not to heavily alter the document, but to make it more consistent and suitable for RAG preprocessing while preserving the original meaning of the chapter content.

In [4]:
def clean_document_text(text: str) -> str:
    """
    Clean extracted PDF text for downstream RAG processing.

    Cleaning steps:
    1. Normalize line breaks
    2. Remove excessive spaces
    3. Remove repeated blank lines
    4. Remove obvious page number-only lines
    5. Strip leading/trailing whitespace
    """
    # Normalize different line break styles
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove lines that contain only page numbers
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)

    # Replace multiple spaces/tabs with a single space
    text = re.sub(r"[ \t]+", " ", text)

    # Reduce excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove extra spaces around line breaks
    text = re.sub(r" *\n *", "\n", text)

    return text.strip()


# Clean the raw text
clean_text = clean_document_text(raw_text)

# Save cleaned text
CLEAN_TEXT_PATH.write_text(clean_text, encoding="utf-8")

print("Cleaned text length:", len(clean_text))
print("First 1000 characters of cleaned text:\n")
print(clean_text[:1000])

Cleaned text length: 91402
First 1000 characters of cleaned text:

Speech and Language Processing.
Daniel Jurafsky & James H. Martin.
Copyright © 2026.
All
rights reserved.
Draft of January 6, 2026.
CHAPTER

Large Language Models
“How much do we know at any time? Much more, or so I believe, than we
know we know.”
Agatha Christie, The Moving Finger
The literature of the fantastic abounds in inanimate objects magically endowed with
the gift of speech. From Ovid’s statue of Pygmalion to Mary Shelley’s story about
Frankenstein, we continually reinvent stories about
creating something and then having a chat with it.
Legend has it that after ﬁnishing his sculpture Moses,
Michelangelo thought it so lifelike that he tapped it
on the knee and commanded it to speak. Perhaps
this shouldn’t be surprising. Language is the mark
of humanity and sentience. conversation is the most
fundamental arena of language, the ﬁrst kind of lan-
guage we learn as children, and the kind we engage in
constantly, whe

## Splitting the Document into Chunks

After cleaning, the chapter text is divided into smaller overlapping chunks. Chunking is a necessary preparation step for RAG because retrieval is performed on smaller text units rather than on the full document at once. In this notebook, a simple character-based chunking strategy is used with overlap between consecutive chunks. The overlap is important because it helps preserve context when a concept spans across chunk boundaries. Each chunk is stored together with a unique `chunk_id`, which will later make it easier to trace retrieved sources during answer generation.

In [5]:
def chunk_text(text: str, chunk_size: int = 1200, overlap: int = 200) -> list[dict]:
    """
    Split text into overlapping character-based chunks.

    Parameters:
        text (str): Full cleaned document text.
        chunk_size (int): Maximum size of each chunk in characters.
        overlap (int): Number of overlapping characters between chunks.

    Returns:
        list[dict]: List of chunk dictionaries with chunk_id and text.
    """
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    chunks = []
    start = 0
    chunk_id = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append({
                "chunk_id": chunk_id,
                "text": chunk
            })
            chunk_id += 1

        # Move forward while keeping overlap
        start += chunk_size - overlap

    return chunks


# Create chunks from cleaned text
chunks = chunk_text(clean_text, chunk_size=1200, overlap=200)

# Save chunks to JSON
CHUNKS_PATH.write_text(json.dumps(chunks, indent=2, ensure_ascii=False), encoding="utf-8")

print("Number of chunks:", len(chunks))
print("\nSample chunk:\n")
pprint(chunks[0])

Number of chunks: 92

Sample chunk:

{'chunk_id': 0,
 'text': 'Speech and Language Processing.\n'
         'Daniel Jurafsky & James H. Martin.\n'
         'Copyright © 2026.\n'
         'All\n'
         'rights reserved.\n'
         'Draft of January 6, 2026.\n'
         'CHAPTER\n'
         '\n'
         'Large Language Models\n'
         '“How much do we know at any time? Much more, or so I believe, than '
         'we\n'
         'know we know.”\n'
         'Agatha Christie, The Moving Finger\n'
         'The literature of the fantastic abounds in inanimate objects '
         'magically endowed with\n'
         'the gift of speech. From Ovid’s statue of Pygmalion to Mary '
         'Shelley’s story about\n'
         'Frankenstein, we continually reinvent stories about\n'
         'creating something and then having a chat with it.\n'
         'Legend has it that after ﬁnishing his sculpture Moses,\n'
         'Michelangelo thought it so lifelike that he tapped it\n'
         'on the

## Inspecting Sample Chunks

Before moving to question-answer generation, it is useful to manually inspect a few chunks. This helps verify that the text cleaning and chunking process has worked properly. At this stage, the chunks should look readable, preserve the intended meaning of the original text, and avoid obvious extraction errors such as broken formatting or meaningless fragments. This manual inspection is also useful for debugging, since poor chunk quality can later reduce retrieval performance.

In [6]:
# Preview a few chunks to inspect quality
for i in range(min(3, len(chunks))):
    print(f"\n--- Chunk {i} ---\n")
    print(chunks[i]["text"][:700])


--- Chunk 0 ---

Speech and Language Processing.
Daniel Jurafsky & James H. Martin.
Copyright © 2026.
All
rights reserved.
Draft of January 6, 2026.
CHAPTER

Large Language Models
“How much do we know at any time? Much more, or so I believe, than we
know we know.”
Agatha Christie, The Moving Finger
The literature of the fantastic abounds in inanimate objects magically endowed with
the gift of speech. From Ovid’s statue of Pygmalion to Mary Shelley’s story about
Frankenstein, we continually reinvent stories about
creating something and then having a chat with it.
Legend has it that after ﬁnishing his sculpture Moses,
Michelangelo thought it so lifelike that he tapped it
on the knee and commanded it to speak. 

--- Chunk 1 ---

our families or friends.
This chapter introduces the Large Language
Model, or LLM, a computational agent that can in-
teract conversationally with people. The fact that LLMs are designed for interaction
with people has strong implications for their design and use

## Preparing Question-Answer Pairs

The assignment requires at least 20 question-answer pairs based strictly on the assigned chapter. These QA pairs serve two purposes. First, the questions act as input queries to the RAG pipelines. Second, the answers act as ground-truth references for evaluation using ROUGE metrics in Task 2. For this reason, the QA pairs must be carefully written so that they are fully supported by the chapter content and not based on outside knowledge. A simple JSON structure is used to store the questions and their corresponding ground-truth answers.

In [7]:
# Template structure for QA pairs
qa_pairs = [
    # 5 Definition Questions
    {
        "question": "What is a Large Language Model (LLM)?",
        "ground_truth_answer": "An LLM is a computational agent or neural network that is designed to interact conversationally with people by predicting the next word from previous words in a given context or prefix. [cite: 13, 60, 78]"
    },
    {
        "question": "What is the definition of 'pretraining' in the context of LLMs?",
        "ground_truth_answer": "Pretraining is the process of learning knowledge about language and the world by iteratively predicting tokens in vast amounts of text. [cite: 41]"
    },
    {
        "question": "What is 'teacher forcing'?",
        "ground_truth_answer": "Teacher forcing is a training approach where the model is always given the correct history sequence to predict the next word, rather than feeding the model its own best guess from the previous time step. [cite: 562]"
    },
    {
        "question": "What is a 'prompt'?",
        "ground_truth_answer": "A prompt is a text string issued by a user to a language model to get the model to do something useful, such as answering a question or following an instruction. [cite: 219]"
    },
    {
        "question": "What is 'perplexity' in language model evaluation?",
        "ground_truth_answer": "Perplexity is a length-normalized metric used to evaluate how well a model predicts unseen text; it is the inverse probability a model assigns to a test set, normalized by the test set length. [cite: 701, 702]"
    },
    # 5 Explanation Questions
    {
        "question": "How does temperature sampling work?",
        "ground_truth_answer": "It reshapes the probability distribution by dividing the logits by a temperature parameter (τ) before the softmax; a low τ increases the probability of high-probability tokens, making the model more 'greedy.' [cite: 403, 405, 459]"
    },
    {
        "question": "How do children achieve high rates of vocabulary growth according to the text?",
        "ground_truth_answer": "Research suggests the bulk of vocabulary acquisition happens as a by-product of reading, which is a process of rich contextual processing rather than learning words in isolation. [cite: 32, 33]"
    },
    {
        "question": "How is an LLM turned from a predictive model into a generative one?",
        "ground_truth_answer": "It is turned into a generative model by repeatedly sampling from its output probability distribution and adding each generated token back into the context as a prefix for the next prediction. [cite: 101, 103]"
    },
    {
        "question": "How does 'in-context learning' differ from standard training?",
        "ground_truth_answer": "In-context learning improves performance through the provided context and activations in the network without involving gradient-based updates to the model's underlying parameters. [cite: 263, 265]"
    },
    {
        "question": "How is 'alignment' performed in the three-stage training process?",
        "ground_truth_answer": "The model is trained on preference data (labeled 'accepted' vs. 'rejected' continuations) using reinforcement learning or reward-based algorithms to make it maximally helpful and less harmful. [cite: 514, 516]"
    },
    # 4 Comparison Questions
    {
        "question": "What is the difference between an Encoder and a Decoder architecture?",
        "ground_truth_answer": "Decoders generate novel output tokens one at a time from left-to-right (generative), whereas Encoders produce vector representations for tokens and are typically used for classification rather than generation. [cite: 135, 136, 146, 151]"
    },
    {
        "question": "How does greedy decoding compare to random sampling?",
        "ground_truth_answer": "Greedy decoding always chooses the single most likely token and is deterministic, while random sampling chooses tokens according to their probability distribution, introducing more diversity. [cite: 321, 352, 359]"
    },
    {
        "question": "What is the difference between zero-shot and few-shot prompting?",
        "ground_truth_answer": "Zero-shot prompting provides instructions without labeled examples, while few-shot prompting includes labeled examples (demonstrations) within the prompt to help the model perform the task. [cite: 234, 235]"
    },
    {
        "question": "How do modern LLMs differ from traditional n-gram models?",
        "ground_truth_answer": "N-gram models predict words based only on a handful of previous words (like bigrams or trigrams), whereas LLMs can use contexts of thousands or tens of thousands of words. [cite: 80, 81]"
    },
    # 3 Importance/Use-case Questions
    {
        "question": "Why is the 'system prompt' important for LLM interaction?",
        "ground_truth_answer": "The system prompt is a first instruction that defines the task, role, tone, and overall context for the LM, and it is silently prepended to all user text. [cite: 266, 267]"
    },
    {
        "question": "What is the importance of the 'distributional hypothesis' for LLMs?",
        "ground_truth_answer": "It proposes that meaning can be learned from text based on the complex association of words with their co-occurring words, allowing models to acquire knowledge simply from reading. [cite: 36, 37]"
    },
    {
        "question": "Why is quality filtering necessary for pretraining corpora?",
        "ground_truth_answer": "Filtering removes boilerplate text, adult content, personal identifiable information (PII), and duplicate documents, which generally improves the performance of the language model. [cite: 634, 635, 637]"
    },
    # 3 Limitation/Challenge Questions
    {
        "question": "What is 'hallucination' in the context of LLMs?",
        "ground_truth_answer": "Hallucination is a safety issue where LLMs generate text that is false or incorrect because the training algorithm lacks a mechanism to enforce factuality. [cite: 778, 779]"
    },
    {
        "question": "What is the challenge of 'data contamination' in model evaluation?",
        "ground_truth_answer": "Data contamination occurs when test set data makes its way into the training set, leading the evaluation metrics to overstate the model's actual performance. [cite: 751, 753]"
    },
    {
        "question": "What are the primary ethical concerns regarding pretraining data scraped from the web?",
        "ground_truth_answer": "Key concerns include potential copyright violations, lack of data consent from website owners, privacy issues from leaked PII, and demographic skew. [cite: 647, 649, 652, 655]"
    }
]

# Save initial template
# Note: QA_PATH should be defined as a Path object in your environment
# QA_PATH.write_text(json.dumps(qa_pairs, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"Generated 20 QA pairs based on the textbook content.")

# Save QA pairs to JSON
QA_PATH.write_text(
    json.dumps(qa_pairs, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(f"Saved {len(qa_pairs)} QA pairs to: {QA_PATH}")

Generated 20 QA pairs based on the textbook content.
Saved 20 QA pairs to: artefacts/qa_pairs_chapter_7.json


# Task 2: Naive RAG vs Contextual Retrieval

Task 2 compares two RAG pipelines built over the same Chapter 7 knowledge base.

- **Naive RAG** retrieves directly from the original chunks.
- **Contextual Retrieval** first adds a short LLM-generated context summary to each chunk, then retrieves from the enriched chunks.

To keep the comparison fair, both methods use:
- the same source chapter
- the same QA pairs
- the same TF-IDF retriever family
- the same OpenAI generator model

The only difference is the chunk representation used for retrieval.

## Task 2 Imports and Configuration

The code below is written directly in the notebook so the full Task 2 pipeline is transparent. Comments are included to make each step easy to follow.

In [8]:
# Task 2 imports
import os
import requests
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Reuse the paths created in Task 1.
ANSWER_DIR = PROJECT_ROOT / 'answer'
ANSWER_DIR.mkdir(parents=True, exist_ok=True)

# Task 2 configuration
CHAPTER_NUMBER = 7
CHAPTER_TITLE = 'Large Language Models'
OPENAI_MODEL = 'gpt-4o-mini'
TOP_K = 4

CONTEXTUAL_CHUNKS_PATH = OUTPUT_DIR / f'chapter_{CHAPTER_NUMBER}_contextual_chunks.json'
FULL_RESULTS_PATH = OUTPUT_DIR / f'task2_full_results_chapter_{CHAPTER_NUMBER}.json'
SUBMISSION_PATH = ANSWER_DIR / f'response-st126477-chapter-{CHAPTER_NUMBER}.json'

print('Contextual chunks path:', CONTEXTUAL_CHUNKS_PATH)
print('Full results path:', FULL_RESULTS_PATH)
print('Submission path:', SUBMISSION_PATH)

Contextual chunks path: artefacts/chapter_7_contextual_chunks.json
Full results path: artefacts/task2_full_results_chapter_7.json
Submission path: answer/response-st126477-chapter-7.json


## Helper Functions for Task 2

These helpers keep the implementation readable while staying fully inside the notebook.

- `.env` loading lets the notebook read `OPENAI_API_KEY`
- `chat_completion()` sends a plain HTTP request to OpenAI
- retrieval uses `TfidfVectorizer` from scikit-learn
- ROUGE is implemented manually so no extra evaluation package is needed

In [9]:
def load_env_file(env_path: Path = PROJECT_ROOT / '.env') -> None:
    """Load variables from a simple .env file if they are not already in the environment."""
    if not env_path.exists():
        return

    for raw_line in env_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue

        key, value = line.split('=', 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value


def get_openai_api_key() -> str:
    """Read the OpenAI API key from the environment after loading .env."""
    load_env_file()
    api_key = os.environ.get('OPENAI_API_KEY', '').strip()
    if not api_key:
        raise RuntimeError('OPENAI_API_KEY was not found. Put it in .env before running Task 2.')
    return api_key


def chat_completion(messages, model: str = OPENAI_MODEL, temperature: float = 0.0, max_tokens: int = 300) -> str:
    """Call the OpenAI Chat Completions API using plain requests."""
    api_key = get_openai_api_key()
    response = requests.post(
        'https://api.openai.com/v1/chat/completions',
        headers={
            'Authorization': f'Bearer {api_key}',
            'Content-Type': 'application/json',
        },
        json={
            'model': model,
            'messages': messages,
            'temperature': temperature,
            'max_tokens': max_tokens,
        },
        timeout=120,
    )
    response.raise_for_status()
    payload = response.json()
    return payload['choices'][0]['message']['content'].strip()


def load_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))


def save_json(data, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding='utf-8')


def strip_citations(text: str) -> str:
    """Remove citation tags like [cite: 13, 60] before scoring ROUGE."""
    return re.sub(r'\[cite:[^\]]+\]', '', text).strip()


def tokenize(text: str) -> list[str]:
    """Simple tokenization used by the ROUGE implementation."""
    return re.findall(r'[a-z0-9]+', strip_citations(text.lower()))


def rouge_n(prediction: str, reference: str, n: int) -> float:
    pred_tokens = tokenize(prediction)
    ref_tokens = tokenize(reference)
    if len(pred_tokens) < n or len(ref_tokens) < n:
        return 0.0

    def ngrams(tokens, n):
        counts = {}
        for i in range(len(tokens) - n + 1):
            gram = tuple(tokens[i:i+n])
            counts[gram] = counts.get(gram, 0) + 1
        return counts

    pred_counts = ngrams(pred_tokens, n)
    ref_counts = ngrams(ref_tokens, n)
    overlap = sum(min(pred_counts[g], ref_counts.get(g, 0)) for g in pred_counts)
    if overlap == 0:
        return 0.0

    precision = overlap / sum(pred_counts.values())
    recall = overlap / sum(ref_counts.values())
    return (2 * precision * recall) / (precision + recall)


def lcs_length(a: list[str], b: list[str]) -> int:
    table = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            if a[i - 1] == b[j - 1]:
                table[i][j] = table[i - 1][j - 1] + 1
            else:
                table[i][j] = max(table[i - 1][j], table[i][j - 1])
    return table[-1][-1]


def rouge_l(prediction: str, reference: str) -> float:
    pred_tokens = tokenize(prediction)
    ref_tokens = tokenize(reference)
    if not pred_tokens or not ref_tokens:
        return 0.0

    lcs = lcs_length(pred_tokens, ref_tokens)
    if lcs == 0:
        return 0.0

    precision = lcs / len(pred_tokens)
    recall = lcs / len(ref_tokens)
    return (2 * precision * recall) / (precision + recall)


def average(values) -> float:
    values = list(values)
    return sum(values) / len(values) if values else 0.0

## Load Task 1 Artefacts

This cell loads the raw chunks and the QA pairs already produced in Task 1. If the JSON file is missing, it falls back to the in-memory `qa_pairs` variable from the earlier Task 1 cell.

In [10]:
# Load the original chunks from Task 1.
chunks = load_json(CHUNKS_PATH)

# Load QA pairs from file if available; otherwise use the qa_pairs variable already defined above.
if QA_PATH.exists():
    qa_pairs_for_eval = load_json(QA_PATH)
else:
    qa_pairs_for_eval = qa_pairs

print('Number of original chunks:', len(chunks))
print('Number of QA pairs:', len(qa_pairs_for_eval))
print('First question:', qa_pairs_for_eval[0]['question'])

Number of original chunks: 92
Number of QA pairs: 20
First question: What is a Large Language Model (LLM)?


## Build Contextual Chunks

Each chunk is enriched with a short summary generated by the OpenAI model. That summary helps retrieval because it gives the retriever more high-level signals about what the chunk is about.

The results are cached, so this cell only needs to be run once unless you want to regenerate them.

In [11]:
def build_contextualized_chunks(chunks: list[dict], document_text: str, title: str) -> list[dict]:
    contextual_chunks = []
    chapter_excerpt = document_text[:4000]  # keep the prompt compact while still giving chapter-level context

    for chunk in chunks:
        prompt = f"""
Title: {title}

Document excerpt:
{chapter_excerpt}

Chunk:
{chunk['text']}

Provide a brief 1-2 sentence context summary describing what this chunk discusses relative to the whole chapter.
Start the answer with: 'This chunk discusses ...'
""".strip()

        context = chat_completion(
            messages=[{'role': 'user', 'content': prompt}],
            model=OPENAI_MODEL,
            temperature=0.0,
            max_tokens=120,
        )

        contextual_chunks.append({
            'chunk_id': chunk['chunk_id'],
            'text': chunk['text'],
            'context': context,
            'retrieval_text': f"{context}{chunk['text']}"
        })

    return contextual_chunks


if CONTEXTUAL_CHUNKS_PATH.exists():
    contextual_chunks = load_json(CONTEXTUAL_CHUNKS_PATH)
    print('Loaded cached contextual chunks.')
else:
    contextual_chunks = build_contextualized_chunks(chunks, clean_text, CHAPTER_TITLE)
    save_json(contextual_chunks, CONTEXTUAL_CHUNKS_PATH)
    print('Built and saved contextual chunks.')

print('Contextual chunk count:', len(contextual_chunks))
print()
print('Sample contextual chunk:')
print(contextual_chunks[0])

Loaded cached contextual chunks.
Contextual chunk count: 92

Sample contextual chunk:
{'chunk_id': 0, 'text': 'Speech and Language Processing.\nDaniel Jurafsky & James H. Martin.\nCopyright © 2026.\nAll\nrights reserved.\nDraft of January 6, 2026.\nCHAPTER\n\nLarge Language Models\n“How much do we know at any time? Much more, or so I believe, than we\nknow we know.”\nAgatha Christie, The Moving Finger\nThe literature of the fantastic abounds in inanimate objects magically endowed with\nthe gift of speech. From Ovid’s statue of Pygmalion to Mary Shelley’s story about\nFrankenstein, we continually reinvent stories about\ncreating something and then having a chat with it.\nLegend has it that after ﬁnishing his sculpture Moses,\nMichelangelo thought it so lifelike that he tapped it\non the knee and commanded it to speak. Perhaps\nthis shouldn’t be surprising. Language is the mark\nof humanity and sentience. conversation is the most\nfundamental arena of language, the ﬁrst kind of lan-\ngua

## Retrieval and Answer Generation Helpers

These helpers implement the actual RAG logic.

- `retrieve_top_chunks()` uses TF-IDF cosine similarity
- `answer_question()` asks the OpenAI model to answer using only retrieved evidence
- `run_rag_method()` executes one method across all 20 evaluation questions

In [12]:
class TfidfRetriever:
    def __init__(self, documents: list[str]):
        self.vectorizer = TfidfVectorizer(stop_words='english')
        self.matrix = self.vectorizer.fit_transform(documents)

    def search(self, query: str, top_k: int = 4):
        query_vector = self.vectorizer.transform([query])
        scores = (self.matrix @ query_vector.T).toarray().ravel()
        ranked = sorted(enumerate(scores.tolist()), key=lambda x: x[1], reverse=True)
        return ranked[:top_k]


def retrieve_top_chunks(question: str, chunk_records: list[dict], top_k: int = 4, use_contextual_text: bool = False) -> list[dict]:
    # For Naive RAG, retrieval sees the original chunk text.
    # For Contextual Retrieval, it sees the context prefix + chunk text.
    retrieval_docs = []
    for chunk in chunk_records:
        if use_contextual_text:
            retrieval_docs.append(chunk['retrieval_text'])
        else:
            retrieval_docs.append(chunk['text'])

    retriever = TfidfRetriever(retrieval_docs)
    ranked = retriever.search(question, top_k=top_k)

    selected = []
    for idx, score in ranked:
        item = dict(chunk_records[idx])
        item['retrieval_score'] = float(score)
        selected.append(item)
    return selected


def answer_question(question: str, retrieved_chunks: list[dict]) -> str:
    context_blocks = []
    for chunk in retrieved_chunks:
        context_blocks.append(f"[Chunk {chunk['chunk_id']}]\n{chunk['text']}")

    joined_context = '\n\n'.join(context_blocks)
    prompt = (
        'Answer the question using only the retrieved context. '
        'Keep the answer concise, factual, and grounded in the chapter. '
        'Do not invent unsupported details.\n\n'
        f"Question: {question}\n\n"
        f"Retrieved context:\n{joined_context}"
    )

    return chat_completion(
        messages=[{'role': 'user', 'content': prompt}],
        model=OPENAI_MODEL,
        temperature=0.0,
        max_tokens=220,
    )


def run_rag_method(qa_pairs: list[dict], chunk_records: list[dict], answer_key: str, use_contextual_text: bool) -> list[dict]:
    rows = []
    for item in qa_pairs:
        retrieved = retrieve_top_chunks(
            question=item['question'],
            chunk_records=chunk_records,
            top_k=TOP_K,
            use_contextual_text=use_contextual_text,
        )
        answer = answer_question(item['question'], retrieved)
        rows.append({
            'question': item['question'],
            'ground_truth_answer': item['ground_truth_answer'],
            answer_key: answer,
            'retrieved_chunk_ids': [chunk['chunk_id'] for chunk in retrieved],
        })
    return rows

## Run Naive RAG

Naive RAG retrieves only from the original chunk text. This is the baseline method.

In [13]:
naive_rows = run_rag_method(
    qa_pairs=qa_pairs_for_eval,
    chunk_records=chunks,
    answer_key='naive_rag_answer',
    use_contextual_text=False,
)

print('Naive RAG answers generated:', len(naive_rows))
print()
print('Sample naive result:')
print(naive_rows[0])

Naive RAG answers generated: 20

Sample naive result:
{'question': 'What is a Large Language Model (LLM)?', 'ground_truth_answer': 'An LLM is a computational agent or neural network that is designed to interact conversationally with people by predicting the next word from previous words in a given context or prefix. [cite: 13, 60, 78]', 'naive_rag_answer': 'A Large Language Model (LLM) is a neural network that predicts the next word based on a given context or prefix of words, outputting a probability distribution over possible next words. It is a larger version of traditional language models and is used for conditional text generation.', 'retrieved_chunk_ids': [40, 14, 6, 69]}


## Run Contextual Retrieval

Contextual Retrieval uses the enriched chunk representation for retrieval, but the same answer-generation model.

In [14]:
contextual_rows = run_rag_method(
    qa_pairs=qa_pairs_for_eval,
    chunk_records=contextual_chunks,
    answer_key='contextual_retrieval_answer',
    use_contextual_text=True,
)

print('Contextual Retrieval answers generated:', len(contextual_rows))
print()
print('Sample contextual result:')
print(contextual_rows[0])

Contextual Retrieval answers generated: 20

Sample contextual result:
{'question': 'What is a Large Language Model (LLM)?', 'ground_truth_answer': 'An LLM is a computational agent or neural network that is designed to interact conversationally with people by predicting the next word from previous words in a given context or prefix. [cite: 13, 60, 78]', 'contextual_retrieval_answer': 'A Large Language Model (LLM) is a neural network that predicts the next word in a sequence based on a given context or prefix. It outputs a probability distribution over possible next words, allowing for the generation of text conditioned on input prompts.', 'retrieved_chunk_ids': [14, 40, 6, 75]}


## Merge Results and Compute ROUGE

This cell combines the two methods into the required assignment format and computes ROUGE-1, ROUGE-2, and ROUGE-L.

In [15]:
# Merge the two method outputs question by question.
final_rows = []
for naive_row, contextual_row in zip(naive_rows, contextual_rows):
    final_rows.append({
        'question': naive_row['question'],
        'ground_truth_answer': naive_row['ground_truth_answer'],
        'naive_rag_answer': naive_row['naive_rag_answer'],
        'contextual_retrieval_answer': contextual_row['contextual_retrieval_answer'],
        'naive_retrieved_chunk_ids': naive_row['retrieved_chunk_ids'],
        'contextual_retrieved_chunk_ids': contextual_row['retrieved_chunk_ids'],
    })

# Compute macro-average ROUGE scores.
naive_r1, naive_r2, naive_rl = [], [], []
contextual_r1, contextual_r2, contextual_rl = [], [], []

for row in final_rows:
    reference = row['ground_truth_answer']
    naive_answer = row['naive_rag_answer']
    contextual_answer = row['contextual_retrieval_answer']

    naive_r1.append(rouge_n(naive_answer, reference, 1))
    naive_r2.append(rouge_n(naive_answer, reference, 2))
    naive_rl.append(rouge_l(naive_answer, reference))

    contextual_r1.append(rouge_n(contextual_answer, reference, 1))
    contextual_r2.append(rouge_n(contextual_answer, reference, 2))
    contextual_rl.append(rouge_l(contextual_answer, reference))

metrics = {
    'naive_rag': {
        'rouge_1': average(naive_r1),
        'rouge_2': average(naive_r2),
        'rouge_l': average(naive_rl),
    },
    'contextual_retrieval': {
        'rouge_1': average(contextual_r1),
        'rouge_2': average(contextual_r2),
        'rouge_l': average(contextual_rl),
    },
}

metrics_df = pd.DataFrame([
    {
        'Method': 'Naive RAG',
        'ROUGE-1': metrics['naive_rag']['rouge_1'],
        'ROUGE-2': metrics['naive_rag']['rouge_2'],
        'ROUGE-L': metrics['naive_rag']['rouge_l'],
    },
    {
        'Method': 'Contextual Retrieval',
        'ROUGE-1': metrics['contextual_retrieval']['rouge_1'],
        'ROUGE-2': metrics['contextual_retrieval']['rouge_2'],
        'ROUGE-L': metrics['contextual_retrieval']['rouge_l'],
    },
])

metrics_df

,Method,ROUGE-1,ROUGE-2,ROUGE-L
0,Naive RAG,0.442479,0.225542,0.331522
1,Contextual Retrieval,0.428664,0.217696,0.335539


## Explanation of the ROUGE Comparison

The table above compares Naive RAG and Contextual Retrieval using three overlap-based metrics:

- **ROUGE-1** measures unigram overlap, so it reflects how well the answer captures the important content words.
- **ROUGE-2** measures bigram overlap, so it is stricter and reflects whether the answer preserves short phrase structure from the reference answer.
- **ROUGE-L** measures longest common subsequence overlap, so it reflects how well the answer preserves the overall sequence and phrasing of the reference answer.

In general, if Contextual Retrieval scores higher than Naive RAG, that suggests the chunk-level context summaries helped retrieval return more relevant evidence. Better retrieval usually leads to answers that are more aligned with the chapter and closer to the ground truth.

A useful way to interpret the results is:

- Higher **ROUGE-1** means the method captured more of the key concepts and vocabulary.
- Higher **ROUGE-2** means the method produced more precise wording and phrase-level alignment.
- Higher **ROUGE-L** means the answer followed the reference answer more coherently.

If Contextual Retrieval improves only slightly, it may mean the original chunks were already strong enough for retrieval. If it improves clearly across all three metrics, it supports the argument that adding context before retrieval helps the model locate better evidence from the chapter.

In [16]:
# Generate a short write-up automatically using the actual scores.
naive = metrics['naive_rag']
contextual = metrics['contextual_retrieval']

comparison_lines = []
for metric_name in ['rouge_1', 'rouge_2', 'rouge_l']:
    naive_score = naive[metric_name]
    contextual_score = contextual[metric_name]
    diff = contextual_score - naive_score
    label = metric_name.replace('_', '-').upper()

    if diff > 0:
        comparison_lines.append(
            f"Contextual Retrieval outperformed Naive RAG on {label} by {diff:.4f} "
            f"({contextual_score:.4f} vs {naive_score:.4f}), suggesting better grounding for this metric."
        )
    elif diff < 0:
        comparison_lines.append(
            f"Naive RAG outperformed Contextual Retrieval on {label} by {abs(diff):.4f} "
            f"({naive_score:.4f} vs {contextual_score:.4f}), suggesting the added chunk context did not help this metric."
        )
    else:
        comparison_lines.append(
            f"Naive RAG and Contextual Retrieval produced the same {label} score ({naive_score:.4f})."
        )

comparison_text = ''.join(comparison_lines)
print(comparison_text)

Naive RAG outperformed Contextual Retrieval on ROUGE-1 by 0.0138 (0.4425 vs 0.4287), suggesting the added chunk context did not help this metric.Naive RAG outperformed Contextual Retrieval on ROUGE-2 by 0.0078 (0.2255 vs 0.2177), suggesting the added chunk context did not help this metric.Contextual Retrieval outperformed Naive RAG on ROUGE-L by 0.0040 (0.3355 vs 0.3315), suggesting better grounding for this metric.


## Save the Assignment Deliverables

The first JSON keeps extra debugging information such as retrieved chunk ids. The second JSON follows the exact assignment submission schema.

In [17]:
# Save the richer debugging artefact.
save_json(final_rows, FULL_RESULTS_PATH)

# Save only the four required fields for submission.
submission_rows = [
    {
        'question': row['question'],
        'ground_truth_answer': row['ground_truth_answer'],
        'naive_rag_answer': row['naive_rag_answer'],
        'contextual_retrieval_answer': row['contextual_retrieval_answer'],
    }
    for row in final_rows
]

save_json(submission_rows, SUBMISSION_PATH)

print('Saved full results to:', FULL_RESULTS_PATH)
print('Saved submission JSON to:', SUBMISSION_PATH)
pd.DataFrame(submission_rows).head()

Saved full results to: artefacts/task2_full_results_chapter_7.json
Saved submission JSON to: answer/response-st126477-chapter-7.json


,question,ground_truth_answer,naive_rag_answer,contextual_retrieval_answer
0,What is a Large Language Model (LLM)?,An LLM is a computational agent or neural netw...,A Large Language Model (LLM) is a neural netwo...,A Large Language Model (LLM) is a neural netwo...
1,What is the definition of 'pretraining' in the...,Pretraining is the process of learning knowled...,"In the context of LLMs, 'pretraining' refers t...",In the context of large language models (LLMs)...
2,What is 'teacher forcing'?,Teacher forcing is a training approach where t...,"Teacher forcing is a training technique where,...","Teacher forcing is a training technique where,..."
3,What is a 'prompt'?,A prompt is a text string issued by a user to ...,A 'prompt' is a text string issued by a user t...,A 'prompt' is a text string issued by a user t...
4,What is 'perplexity' in language model evaluat...,Perplexity is a length-normalized metric used ...,Perplexity in language model evaluation is a m...,Perplexity in language model evaluation is a m...
